In [1]:
"""
RAG 기반 임상시험 프로토콜 생성 및 성능평가
- 생성된 프로토콜(A)과 검색된 실제 프로토콜(B_1~B_N) 비교
- METEOR, ROUGE 점수로 생성 품질 평가
- 사용자 입력: 질환명 + 연구제목
- 모델 : meta-llama/Llama-3.1-8B-Instruct
"""

#---------- 1. 환경설정
import os 
import json
import time
import pickle 
import random
import numpy as np
from dotenv import load_dotenv
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, pipeline, set_seed
import faiss
import re

# 평가 라이브러리
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
import nltk

try:
    nltk.data.find('wordnet')
    nltk.data.find('omw-1.4') 
except LookupError:
    nltk.download('wordnet')
    nltk.download('omw-1.4') 
    nltk.download('punkt')
    nltk.download('punkt_tab')

# env 파일로부터 HuggingFace 토큰 읽어오기
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

# 경로 설정
INDEX_PATH = "faiss_index"
PARSED_DATA_PATH = "parsed_data/parsed_ctg-studies.json"

# 모델 설정
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-4B"
LLM_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
# LLM_MODEL = "google/gemma-2-9b"
# LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"

# GPU 설정
EMBEDDING_DEVICE = "cuda:0"
LLM_DEVICE = "cuda:1"

# 검색 설정 (Few-shot용)
TOP_K = 3

# 평가용 비교 대상 수
TOP_K_EVAL = 5  # 생성된 A와 비교할 실제 프로토콜 B 개수

# 시드 설정
SEED = 42


#---------- 2. 시드 고정
def set_all_seeds(seed: int = SEED):
    # 재현성을 위한 시드 고정
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)  
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    print("Seed 고정",seed)


#---------- 3. 평가 메트릭 클래스
class ProtocolEvaluator:
    # METEOR, ROUGE 점수 계산 클래스
    
    def __init__(self):
        # ROUGE 스코어 초기화 (rouge1, rouge2, rougeL)
        self.rouge = rouge_scorer.RougeScorer(
            ['rouge1', 'rouge2', 'rougeL'], 
            use_stemmer=True
        )
    
    def protocol_to_text(self, protocol: dict) -> str:
        # 프로토콜 JSON을 평가용 텍스트로 변환
        # Study Overview, Participation Criteria, Study Plan에서 주요 필드 추출
        texts = []
        
        # Study Overview 추출
        if "Study Overview" in protocol:
            overview = protocol["Study Overview"]
            if overview.get("Brief Title"):
                texts.append("Title: " + str(overview.get("Brief Title", "")))
            if overview.get("Brief Summary"):
                texts.append("Summary: " + str(overview.get("Brief Summary", "")))
            if overview.get("Conditions"):
                conditions = overview.get("Conditions", [])
                if isinstance(conditions, list):
                    texts.append("Conditions: " + ", ".join(conditions))
                else:
                    texts.append("Conditions: " + str(conditions))
            if overview.get("Study Type"):
                texts.append("Study Type: " + str(overview.get("Study Type", "")))
            if overview.get("Interventions"):
                interventions = overview.get("Interventions", [])
                for inv in interventions:
                    if isinstance(inv, dict):
                        texts.append("Intervention: " + str(inv.get("Name", "")))
        
        # Participation Criteria 추출
        if "Participation Criteria" in protocol:
            criteria = protocol["Participation Criteria"]
            if criteria.get("Eligibility Criteria"):
                texts.append("Eligibility: " + str(criteria.get("Eligibility Criteria", "")))
            if criteria.get("Sex"):
                texts.append("Sex: " + str(criteria.get("Sex", "")))
            if criteria.get("Minimum Age"):
                texts.append("Min Age: " + str(criteria.get("Minimum Age", "")))
            if criteria.get("Maximum Age"):
                texts.append("Max Age: " + str(criteria.get("Maximum Age", "")))
        
        # Study Plan 추출
        if "Study Plan" in protocol:
            plan = protocol["Study Plan"]
            design = plan.get("Design Information", {})
            if design.get("Allocation"):
                texts.append("Allocation: " + str(design.get("Allocation", "")))
            if design.get("Masking"):
                texts.append("Masking: " + str(design.get("Masking", "")))
            if design.get("Primary Purpose"):
                texts.append("Purpose: " + str(design.get("Primary Purpose", "")))
            
            # Primary Outcomes
            outcomes = plan.get("Primary Outcomes", [])
            for outcome in outcomes:
                if isinstance(outcome, dict):
                    texts.append("Outcome: " + str(outcome.get("Measure", "")))
        
        return " ".join(texts)
    
    def calculate_meteor(self, reference: str, hypothesis: str) -> float:
        # METEOR 점수 계산
        # 이때 동의어, 어간 변화를 고려한 유사도 측정
        ref_tokens = reference.lower().split()
        hyp_tokens = hypothesis.lower().split()
        
        if len(ref_tokens) == 0 or len(hyp_tokens) == 0:
            return 0.0
        
        try:
            score = meteor_score([ref_tokens], hyp_tokens)
        except:
            score = 0.0
        
        return score
    
    def calculate_rouge(self, reference: str, hypothesis: str) -> dict:
        # ROUGE 점수 계산
        # N-gram 중복 기반 유사도 측정
        if not reference or not hypothesis:
            return {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}
        
        try:
            scores = self.rouge.score(reference, hypothesis)
            return {
                "rouge1": scores["rouge1"].fmeasure,
                "rouge2": scores["rouge2"].fmeasure,
                "rougeL": scores["rougeL"].fmeasure
            }
        except:
            return {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}
    
    def evaluate(self, generated: dict, reference: dict) -> dict:
        # 생성된 프로토콜(A)과 참조 프로토콜(B) 비교 및 종합 평가
        gen_text = self.protocol_to_text(generated)
        ref_text = self.protocol_to_text(reference)
        
        meteor = self.calculate_meteor(ref_text, gen_text)
        rouge = self.calculate_rouge(ref_text, gen_text)
        
        return {
            "meteor": meteor,
            "rouge1": rouge["rouge1"],
            "rouge2": rouge["rouge2"],
            "rougeL": rouge["rougeL"],
            "generated_text_length": len(gen_text),
            "reference_text_length": len(ref_text)
        }


#---------- 4. RAG 구현
class ClinicalTrialRAG:
    # RAG 기반 임상시험 프로토콜 생성 클래스
    
    def __init__(self):
        self.index = None # FAISS 인덱스 객체
        self.nct_ids = None # NCT ID 리스트 (index -> NCT ID 매핑)
        self.study_data = None # 전체 임상시험 데이터 딕셔너리 
        self.embed_tokenizer = None # 임베딩 모델 토크나이저
        self.embed_model = None # 임베딩 모델
        self.llm_pipe = None # LLM 파이프 라인
        self.llm_tokenizer = None # LLM 토크나이저
        self.llm_model = None # LLM 모델
    
    def load_faiss_index(self):
        # FAISS 인덱스 및 메타데이터 로딩
        start = time.time()
        print("FAISS 인덱스 로딩 시작")
        
        # FAISS 인덱스 로드 (벡터 데이터)
        self.index = faiss.read_index(os.path.join(INDEX_PATH, "clinical_trials.index"))

        # NCT ID 리스트 로드 (인덱스 번호 -> NCT ID 변환용)
        with open(os.path.join(INDEX_PATH, "nct_ids.pkl"), 'rb') as f:
            self.nct_ids = pickle.load(f)

        # 전체 임상시험 데이터 로드 (NCT ID -> 임상시험 프로토콜 데이터 상세 정보)
        with open(os.path.join(INDEX_PATH, "study_data.pkl"), 'rb') as f:
            self.study_data = pickle.load(f)
            
        elapsed = time.time() - start
        print("FAISS 인덱스 로딩 완료:", self.index.ntotal, "개,", round(elapsed, 1), "초")
    
    def load_embedding_model(self):
        # 임베딩 모델 로딩
        start = time.time()
        print("임베딩 모델 로딩 시작")

        # 텍스트를 토큰 ID로 변환하는 토크나이저 로딩
        self.embed_tokenizer = AutoTokenizer.from_pretrained(
            EMBEDDING_MODEL, 
            trust_remote_code=True
        )

        # 토큰을 벡터로 변환하는 모델 로딩
        self.embed_model = AutoModel.from_pretrained(
            EMBEDDING_MODEL,
            torch_dtype=torch.float16,
            trust_remote_code=True
        ).to(EMBEDDING_DEVICE)
        self.embed_model.eval()
        
        elapsed = time.time() - start
        print("임베딩 모델 로딩 완료:", round(elapsed, 1), "초")
    
    def load_llm(self):
        # LLM 로딩
        start = time.time()
        print("LLM 로딩 시작")
        
        self.llm_tokenizer = AutoTokenizer.from_pretrained(
            LLM_MODEL,
            token=HF_TOKEN,
            trust_remote_code=True
        )
        
        # GPU ID 호출
        gpu_id = int(LLM_DEVICE.split(":")[-1])
        
        self.llm_model = AutoModelForCausalLM.from_pretrained(
            LLM_MODEL,
            token=HF_TOKEN,
            torch_dtype=torch.float16,
            device_map={"": gpu_id},  # int로 전달해야 GPU 활용 가능
            trust_remote_code=True
        )

        # pipeline 생성 
        self.llm_pipe = pipeline(
            "text-generation",
            model=self.llm_model,
            tokenizer=self.llm_tokenizer,
            torch_dtype=torch.float16,
        )
        
        elapsed = time.time() - start
        print("LLM 로딩 완료:", round(elapsed, 1), "초")
    
    def load_all(self):
        # FAISS 인덱스, 임베딩 모델, LLM 모델 로딩
        self.load_faiss_index()
        self.load_embedding_model()
        self.load_llm()
    
    def get_query_embedding(self, query: str) -> np.ndarray:
        # 사용자 입력 쿼리를 벡터로 변환
        with torch.no_grad():
            inputs = self.embed_tokenizer(
                query,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            ).to(EMBEDDING_DEVICE)
            
            outputs = self.embed_model(**inputs)
            
            # Mean pooling 적용
            attention_mask = inputs["attention_mask"]
            token_embeddings = outputs.last_hidden_state
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
            embeddings = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            
            # L2 정규화 적용
            embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        
        return embeddings.cpu().numpy().astype('float32')
    
    def search_similar(self, condition: str, title: str, top_k: int) -> list:
        # 질환명 + 연구제목으로 유사 임상시험 검색
        # 인덱싱 형식에 맞춰 쿼리 생성
        start = time.time()
        query = "Title: " + title + " | Conditions: " + condition
        query_embedding = self.get_query_embedding(query)
        
        scores, indices = self.index.search(query_embedding, top_k)
        
        results = []
        for i, idx in enumerate(indices[0]):
            nct_id = self.nct_ids[idx]
            study = self.study_data[nct_id]
            results.append({
                "nct_id": nct_id,
                "score": float(scores[0][i]),
                "data": study
            })
        
        elapsed = time.time() - start
        print("유사 임상시험 검색 완료:", round(elapsed, 2), "초")
        for r in results:
            r_title = r["data"]["Study Overview"].get("Brief Title", "N/A")[:60]
            print("  -", r['nct_id'], ":", r_title, "... (유사도:", round(r['score'], 4), ")")
        
        return results
    
    def build_rag_prompt(self, condition: str, title: str, similar_studies: list) -> str:
        # RAG 기반 Few-Shot CoT 프롬프트 구성

        # 시스템 프롬프트로 역할 및 지시사항 정의
        system_prompt = """You are an expert clinical trial protocol designer with extensive experience in regulatory compliance and study design.

## Your Role:
- Synthesize information from similar clinical trials and user requirements to generate a complete, scientifically rigorous clinical trial protocol
- Apply your expertise in clinical research methodology, biostatistics, and regulatory guidelines (ICH-GCP, FDA, EMA)

## Task Instructions:
1. **Analyze Similar Studies**: Review the provided reference clinical trials to understand established patterns and best practices for the given condition/intervention
2. **Synthesize User Requirements**: Combine user input with insights from similar studies to design an appropriate protocol
3. **Step-by-Step Reasoning**: Explain your reasoning process for each design decision
4. **Generate Complete Protocol**: Produce a comprehensive protocol with three main sections:
   - **Study Overview**: Title, summary, conditions, study type, phases, sponsor, interventions
   - **Participation Criteria**: Inclusion/exclusion criteria, age, sex, volunteer status
   - **Study Plan**: Design information, arm groups, primary/secondary outcomes

## Output Requirements:
- Reasoning process: Detailed explanation
- Final JSON output: ALL field values MUST be in English
- Follow the exact JSON structure shown in the examples
- Ensure scientific accuracy and regulatory compliance"""

        # 프롬프트 형식 정의
        prompt = "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n" + system_prompt + "<|eot_id|>"
               
        # Few-shot 예시 
        for i, study in enumerate(similar_studies, 1):
            nct_id = study["nct_id"]
            data = study["data"]
            overview = data.get("Study Overview", {})
            
            # 예시 입력 구성
            example_conditions = overview.get('Conditions', [])
            example_title = overview.get('Brief Title', 'N/A')
            example_interventions = [inv.get('Name', '') for inv in overview.get('Interventions', [])]
            
            example_input = """
Reference Clinical Trial #""" + str(i) + """ (""" + nct_id + """):
- Condition: """ + ', '.join(example_conditions) + """
- Title: """ + example_title + """
- Study Type: """ + str(overview.get('Study Type', 'N/A')) + """
- Interventions: """ + ', '.join(example_interventions)
            
            # 예시 출력 (실제 데이터)
            example_output = json.dumps(data, ensure_ascii=False, indent=2)
            
            # 추론 과정 생성
            reasoning = self._generate_reasoning(data)
            
            prompt += """
<|start_header_id|>user<|end_header_id|>

""" + example_input + """<|eot_id|>

<|start_header_id|>assistant<|end_header_id|>

""" + reasoning + """

Final Output:
```json
""" + example_output + """
```<|eot_id|>
"""
        
        # 실제 사용자 입력 프롬프트
        user_prompt = """
## User Request:
- Condition: """ + condition + """
- Study Title: """ + title + """

## Instructions for This Request:
1. Analyze the above research requirements and the """ + str(len(similar_studies)) + """ reference clinical trials provided
2. Identify key design elements: study type, intervention type, target population, primary endpoints
3. Explain your reasoning process step-by-step for each protocol section
4. Generate a complete clinical trial protocol in JSON format

Please proceed with your analysis and protocol generation."""

        prompt += """
<|start_header_id|>user<|end_header_id|>

""" + user_prompt + """<|eot_id|>

<|start_header_id|>assistant<|end_header_id|>

## Reasoning Process:

### 1. Study Overview Analysis:
"""
        
        return prompt

    def _generate_reasoning(self, data: dict) -> str:
        # 추론과정 템플릿 생성
        overview = data.get("Study Overview", {})
        criteria = data.get("Participation Criteria", {})
        plan = data.get("Study Plan", {})
        
        study_type = overview.get("Study Type", "N/A")
        phases = overview.get("Phases", [])
        conditions = overview.get("Conditions", [])
        interventions = overview.get("Interventions", [])
        allocation = plan.get("Design Information", {}).get("Allocation", "N/A")
        masking = plan.get("Design Information", {}).get("Masking", "N/A")
        purpose = plan.get("Design Information", {}).get("Primary Purpose", "N/A")
        
        intervention_types = [inv.get("Type", "") for inv in interventions]
        intervention_names = [inv.get("Name", "") for inv in interventions]
        
        # 추론 과정 문자열 생성
        reasoning = """## Reasoning Process:

### 1. Study Overview Analysis:
- Target Condition: """ + ', '.join(conditions) + """
- Intervention: """ + ', '.join(intervention_names) + """ (""" + ', '.join(intervention_types) + """ type)
- Study Type Decision: """ + study_type + """ is appropriate as active intervention is performed on participants
- Clinical Trial Phase: """ + (', '.join(phases) if phases else 'NA') + """

### 2. Participation Criteria Analysis:
- Target Population: Patients with """ + ', '.join(conditions) + """
- Healthy Volunteers: """ + str(criteria.get('Healthy Volunteers', 'N/A')) + """
- Age Range: """ + str(criteria.get('Minimum Age', 'N/A')) + """ to """ + str(criteria.get('Maximum Age', 'N/A')) + """
- Sex: """ + str(criteria.get('Sex', 'N/A')) + """

### 3. Study Plan Analysis:
- Primary Purpose: """ + purpose + """
- Allocation: """ + allocation + """
- Masking: """ + masking + """
- Intervention Model: """ + str(plan.get("Design Information", {}).get("Intervention Model", "N/A")) + """

### 4. Conclusion:
Based on the above analysis, this protocol follows """ + study_type + """ design with """ + allocation + """ allocation, """ + masking + """ masking, and """ + purpose + """ as the primary purpose.

Final Output:"""
        
        return reasoning
    
    def generate_protocol(self, condition: str, title: str, max_new_tokens: int = 2048) -> dict:
        # 생성함수

        set_all_seeds(SEED)
        
        # 유사 임상시험 검색
        print("\n 1. 유사 임상시험 검색")
        similar_studies = self.search_similar(condition, title, TOP_K)
        
        # RAG 프롬프트 구성
        print("\n 2. 프롬프트 구성")
        prompt = self.build_rag_prompt(condition, title, similar_studies)
        
        # LLM 생성
        print("\n 3. 프로토콜 생성")
        start = time.time()
        
        outputs = self.llm_pipe(
            prompt,
            max_new_tokens=max_new_tokens, # 최대 생성 토큰 수
            do_sample=True, #  샘플링 활성화
            temperature=0.6, 
            top_p=0.9, # 누적확률 정의
            repetition_penalty=1.1, # 반복 방지
            return_full_text=False
        )
        
        generated_text = outputs[0]["generated_text"]
        elapsed = time.time() - start
        
        if elapsed >= 60:
            print("생성 완료:", int(elapsed // 60), "분", int(elapsed % 60), "초")
        else:
            print("생성 완료:", round(elapsed, 1), "초")

        # JSON 추출
        result = {}
        reasoning_text = ""
        
        try:
            json_start = generated_text.find("```json")
            if json_start != -1:
                reasoning_text = generated_text[:json_start].strip()
                json_end = generated_text.find("```", json_start + 7)
                if json_end != -1:
                    json_str = generated_text[json_start + 7:json_end].strip()
                    result = json.loads(json_str)
            else:
                json_start = generated_text.find("{")
                if json_start != -1:
                    reasoning_text = generated_text[:json_start].strip()
                    json_end = generated_text.rfind("}") + 1
                    json_str = generated_text[json_start:json_end]
                    result = json.loads(json_str)
                    
        except json.JSONDecodeError:
            print("JSON 파싱 실패")
            result = {"raw_output": generated_text}
        
        return result, generated_text, similar_studies, reasoning_text
    
    def release_gpu(self):
        # GPU 메모리 해제
        if self.embed_model is not None:
            self.embed_model.cpu()
            del self.embed_model
        if self.embed_tokenizer is not None:
            del self.embed_tokenizer
        if self.llm_model is not None:
            self.llm_model.cpu()
            del self.llm_model
        if self.llm_pipe is not None:
            del self.llm_pipe
        if self.llm_tokenizer is not None:
            del self.llm_tokenizer
        
        import gc
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        print("GPU 메모리 해제 완료")


#---------- 6. 메인 실행
if __name__ == "__main__":
    # 사용자 입력: 질환명 + 연구제목
    
    condition = "Pulmonary Embolism"
    title = "Diagnostic Utility of Point-of-Care Ultrasound in Suspected Pulmonary Embolism"

    print("RAG 기반 임상시험 프로토콜 생성 및 성능 평가")
    print("입력 조건:")
    print(" 1. Condition:", condition)
    print(" 2. Title:", title)
    print("평가 방식: 생성된 A vs 검색된 B_1~B_" + str(TOP_K_EVAL) + " 비교")
    
    # Seed 고정
    set_all_seeds(SEED)
    
    # RAG 시스템 및 평가 클래스 초기화
    rag = ClinicalTrialRAG()
    evaluator = ProtocolEvaluator()
    
    # 결과 저장용 딕셔너리
    evaluation_result = {}
    
    try:
        # 모델 로딩
        rag.load_all()
        
        # 1단계 : 프로토콜 생성 
        print("1단계 : 프로토콜 생성")
        
        start_time = time.time()
        generated_protocol, raw_output, fewshot_studies, reasoning = rag.generate_protocol(
            condition, 
            title
        )
        generation_time = time.time() - start_time

        print("\n생성 시간:", round(generation_time, 1), "초")
        
        # 2단계 : 비교 대상 검색 (B_1 ~ B_N)
        print("2. 비교 대상 검색 (B_1 ~ B_" + str(TOP_K_EVAL) + ")")

        
        reference_studies = rag.search_similar(condition, title, TOP_K_EVAL)
        
        print("\n검색된 비교 대상 프로토콜:")
        for i, ref in enumerate(reference_studies):
            ref_title = ref["data"]["Study Overview"].get("Brief Title", "N/A")[:50]
            print("  B_" + str(i+1) + ":", ref["nct_id"], "-", ref_title, "... (유사도:", round(ref["score"], 4), ")")
        
        # 3단계 : A, B 비교 및 평가
        print("3단계 : 평가 (A vs B)")
        
        comparisons = []
        
        for i, ref_study in enumerate(reference_studies):
            ref_nct_id = ref_study["nct_id"]
            ref_data = ref_study["data"]
            ref_score = ref_study["score"]
            ref_title = ref_data["Study Overview"].get("Brief Title", "N/A")
            
            # A vs B_i 평가
            scores = evaluator.evaluate(generated_protocol, ref_data)
            
            comparison = {
                "reference_id": "B_" + str(i+1),
                "reference_nct_id": ref_nct_id,
                "reference_title": ref_title,
                "similarity_score": ref_score,
                "meteor": scores["meteor"],
                "rouge1": scores["rouge1"],
                "rouge2": scores["rouge2"],
                "rougeL": scores["rougeL"],
                "generated_text_length": scores["generated_text_length"],
                "reference_text_length": scores["reference_text_length"]
            }
            comparisons.append(comparison)
            
            print("\nA vs B_" + str(i+1) + " (" + ref_nct_id + "):")
            print("  제목:", ref_title[:60], "..." if len(ref_title) > 60 else "")
            print("  METEOR:", round(scores["meteor"], 4))
            print("  ROUGE-1:", round(scores["rouge1"], 4))
            print("  ROUGE-2:", round(scores["rouge2"], 4))
            print("  ROUGE-L:", round(scores["rougeL"], 4))
        
        # 평균 점수 계산
        avg_meteor = np.mean([c["meteor"] for c in comparisons])
        avg_rouge1 = np.mean([c["rouge1"] for c in comparisons])
        avg_rouge2 = np.mean([c["rouge2"] for c in comparisons])
        avg_rougeL = np.mean([c["rougeL"] for c in comparisons])
        
        # 결과 요약해서 출력
        print("평가 결과 요약")
        
        print("\n입력 조건:")
        print(" 1. Condition:", condition)
        print(" 2. Title:", title)
        
        print("\n비교 대상 수:", len(comparisons))
        
        print("\n평균 점수:")
        print("  METEOR:", round(avg_meteor, 4))
        print("  ROUGE-1:", round(avg_rouge1, 4))
        print("  ROUGE-2:", round(avg_rouge2, 4))
        print("  ROUGE-L:", round(avg_rougeL, 4))
        
        # 결과 저장
        os.makedirs("Results", exist_ok=True)
        
        evaluation_result = {
            "evaluation_config": {
                "TOP_K": TOP_K,
                "top_k_eval": TOP_K_EVAL,
                "seed": SEED,
                "embedding_model": EMBEDDING_MODEL,
                "llm_model": LLM_MODEL
            },
            "input": {
                "condition": condition,
                "title": title
            },
            "generation_info": {
                "generation_time_seconds": round(generation_time, 2),
                "fewshot_studies": [{"nct_id": s["nct_id"], "score": s["score"]} for s in fewshot_studies]
            },
            "average_scores": {
                "meteor": round(avg_meteor, 4),
                "rouge1": round(avg_rouge1, 4),
                "rouge2": round(avg_rouge2, 4),
                "rougeL": round(avg_rougeL, 4)
            },
            "detailed_comparisons": comparisons,
            "generated_protocol": generated_protocol,
            "reasoning": reasoning
        }
        
        result_path = "Results/Llama_evaluation_results.json"
        with open(result_path, 'w', encoding='utf-8') as f:
            json.dump(evaluation_result, f, ensure_ascii=False, indent=2)
        
        print("\n결과 저장 완료:", result_path)
        
    finally:
        rag.release_gpu()
    

/data/home/bmi-lab/anaconda3/envs/jy3/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package wordnet to
[nltk_data]     /data/home/sss2061/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /data/home/sss2061/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /data/home/sss2061/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /data/home/sss2061/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


RAG 기반 임상시험 프로토콜 생성 및 성능 평가
입력 조건:
 1. Condition: Pulmonary Embolism
 2. Title: Diagnostic Utility of Point-of-Care Ultrasound in Suspected Pulmonary Embolism
평가 방식: 생성된 A vs 검색된 B_1~B_5 비교
Seed 고정 42
FAISS 인덱스 로딩 시작
FAISS 인덱스 로딩 완료: 557387 개, 46.5 초
임베딩 모델 로딩 시작


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  3.14it/s]


임베딩 모델 로딩 완료: 5.3 초
LLM 로딩 시작


Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.25it/s]
`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:1


LLM 로딩 완료: 5.7 초
1. 프로토콜 생성 (A)
Seed 고정 42

 1. 유사 임상시험 검색


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


유사 임상시험 검색 완료: 3.13 초
  - NCT04882579 : Point-of-care Ultrasound in Suspected Pulmonary Embolism ... (유사도: 0.8909 )
  - NCT02483442 : D-dimer Testing Tailored to Clinical Pretest Probability in  ... (유사도: 0.8902 )
  - NCT01558206 : Ultrasonography of Chest Versus Pulmonary Artery CT Angiogra ... (유사도: 0.8879 )

 2. 프롬프트 구성

 3. 프로토콜 생성 중...
생성 완료: 35.7 초

생성 시간: 38.9 초
2. 비교 대상 검색 (B_1 ~ B_5)
유사 임상시험 검색 완료: 2.22 초
  - NCT04882579 : Point-of-care Ultrasound in Suspected Pulmonary Embolism ... (유사도: 0.8909 )
  - NCT02483442 : D-dimer Testing Tailored to Clinical Pretest Probability in  ... (유사도: 0.8902 )
  - NCT01558206 : Ultrasonography of Chest Versus Pulmonary Artery CT Angiogra ... (유사도: 0.8879 )
  - NCT05469724 : Clinical Pulmonary Embolism ... (유사도: 0.8868 )
  - NCT00007085 : Prospective Investigation of Pulmonary Embolism Diagnosis (P ... (유사도: 0.883 )

검색된 비교 대상 프로토콜:
  B_1: NCT04882579 - Point-of-care Ultrasound in Suspected Pulmonary Em ... (유사도: 0.8909 )
  B_2: NCT02483442 - D